In [2]:
from paths import *
from nano_maker.nanomaker import NanoMaker
from nano_maker.container.configs import (skeleton_weights, naanobot_weights,
                                          skeleton_config, naanobot_config, radial_config)

SyntaxError: expected '(' (nanomaker.py, line 107)

In [2]:
skeleton_weight_filename = skeleton_weights
skeleton_cfg = skeleton_config
naanobot_weight_filename = naanobot_weights
naanobot_config = naanobot_config
radial_cfg = radial_config

In [3]:
model = NanoMaker(skeleton_weight_filename=skeleton_weight_filename,
                  naanobot_weight_filename=naanobot_weight_filename,
                  skeleton_cfg=skeleton_config,
                  naanobot_config=naanobot_config,
                  radial_cfg=radial_cfg)

In [4]:
drug_i_want_to_deliver = "CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C"

In [5]:
model.ingest_chemical(drug_i_want_to_deliver)

Chemical Ingested: CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C
Drug Likeness Rules Passed: 4 / 4


In [6]:
pocket_data = model.generate_pocket_data()
print(len(pocket_data))
print(type(pocket_data))

TypeError: unhashable type: 'list'

In [7]:
pocket_data

[[9.995849057766371, -11.445503276060219, -0.48738792713909124],
 [12.144050448136944, 4.438241428197432, -5.872303960968976],
 [9.393153192651333, 5.934358035104158, -8.839636311903774],
 [3.8314803322516657, -11.465542496487348, -6.032985902412316],
 [11.964228129460906, 2.23613557979413, 4.194272532121009],
 [-4.940990651618355, -8.699841325215187, -7.716817910926168],
 [9.342879001837703, -7.943406966168858, -2.5961746984532064],
 [11.651277687027124, -0.6303016076387501, -3.801519479976501],
 [5.737437320975602, 6.162923678151892, -8.455206632589212],
 [-0.7390686667011566, -7.543078080621538, -9.015416136166102],
 [2.7375066675150106, -11.279621797074867, 0.1427847535040686],
 [1.6340438658188168, 10.950934577980398, -2.5841861349192317],
 [2.874170043728385, 6.565924470762589, -8.832099175316069],
 [3.0986952288079985, 9.183976778495063, 5.239955028336741],
 [-3.9537120815972573, 0.14699165970580616, -10.584665136023158],
 [8.409423723111274, 5.453955478611608, 4.053072095374276

In [ ]:
# GENERATION QUALITY ASSESSMENTS

In [10]:
import torch
from nano_maker.naanobot import NAAnoBot
from nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

@torch.no_grad()
def diagnose_generation(model, map4_enc, sph_coordinates, n=10):
    model.eval()
    map4_enc = map4_enc.to(next(model.parameters()).device)

    bioch_context = [model.naano_module.get_nAAnovector("VOID") for _ in range(model.block_size)]
    coord_context = [[model.max_angstroms, 0, 0] for _ in range(model.block_size)]

    for i, coordinate in enumerate(sph_coordinates[:n]):
        naano_X = model.naano_module.get_nAAno_X(coord_context, bioch_context, coordinate).unsqueeze(0).to(map4_enc.device)
        output, _ = model.forward(naano_X, map4_enc)

        # print raw predicted vector
        print(f"\nPosition {i}:")
        print(f"  raw output: {output[0].detach().cpu().numpy().round(2)}")

        # print distances to all amino acids
        vector = output[0].detach().float()
        distances = {}
        for aa_id, n_v in model.naano_module.nAAno_vectors.items():
            if aa_id == 'VOID':
                continue
            n_v_t = torch.tensor(n_v, dtype=torch.float32)
            distances[aa_id] = torch.norm(vector - n_v_t).item()

        sorted_distances = sorted(distances.items(), key=lambda x: x[1])
        print(f"  top 3: {sorted_distances[:3]}")

        # update context
        aa_id = sorted_distances[0][0]
        bioch_context = bioch_context[1:] + [model.naano_module.get_nAAnovector(aa_id)]
        coord_context = coord_context[1:] + [coordinate.tolist() if torch.is_tensor(coordinate) else coordinate]


map4_enc = torch.tensor(smiles_fingerprint(drug_i_want_to_deliver), dtype=torch.float32).unsqueeze(0)
nb_cfg = naanobot_config.copy()
naanobot_model = NAAnoBot(n_embd=nb_cfg["n_embd"], n_head=nb_cfg["n_head"],
                                           n_layers=nb_cfg["n_layers"],
                                           block_size=nb_cfg["block_size"],
                                           map4_res=nb_cfg["map4_res"],
                                           n_nAAno_features=nb_cfg["n_nAAno_features"],
                                           n_spatial_features=nb_cfg["n_spatial_features"],
                                           max_angstroms=nb_cfg["max_angstroms"],
                                           dropout=nb_cfg["dropout"])
diagnose_generation(model=naanobot_model, map4_enc=map4_enc, sph_coordinates=model._pocket_sph_skeleton())


Position 0:
  raw output: [ 0.41  0.54  0.52  0.13 -0.79  0.67  0.48 -0.04  0.08 -0.47  0.11 -0.96
 -0.03  0.26 -0.06 -0.65  0.17  0.    0.36 -0.82 -0.64]
  top 3: [('A', 24.11823081970215), ('G', 24.43145179748535), ('T', 24.85062599182129)]

Position 1:
  raw output: [-0.45  0.61  0.03  0.47  0.48  1.18 -0.11 -0.19 -0.22  0.32  0.38  0.19
  0.06  0.02 -0.75 -0.57 -0.09 -0.45 -0.3  -0.82  0.04]
  top 3: [('A', 24.72063446044922), ('G', 24.731430053710938), ('T', 25.545949935913086)]

Position 2:
  raw output: [-0.11  0.99  0.12  0.2  -0.01  0.74  0.34  0.35 -0.27  0.39  0.17 -0.12
  0.29  0.3  -0.25 -0.64 -0.33 -0.69  0.06 -0.39  0.01]
  top 3: [('G', 24.2540283203125), ('A', 24.500995635986328), ('T', 25.09638786315918)]

Position 3:
  raw output: [-0.05 -0.02  0.43 -0.07  0.39  0.71  0.03 -0.45  0.05  0.11  0.41  0.33
  0.49 -0.06 -0.98 -0.66 -0.22 -0.15 -0.95 -0.46  0.55]
  top 3: [('G', 24.07340431213379), ('A', 24.30423927307129), ('D', 24.89886474609375)]

Position 4:
  raw out